In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv", nrows=10000)

In [2]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate tweets:")
print(df["tweet_id"].duplicated().sum())

print("\nInbound distribution:")
print(df["inbound"].value_counts())

print("\nUnique authors:")
print(df["author_id"].nunique())

print("\nDate range:")
print(df["created_at"].min())
print(df["created_at"].max())

Rows: 10000
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Missing values:
tweet_id                      0
author_id                     0
inbound                       0
created_at                    0
text                          0
response_tweet_id          3281
in_response_to_tweet_id    2731
dtype: int64

Duplicate tweets:
0

Inbound distribution:
inbound
True     5503
False    4497
Name: count, dtype: int64

Unique authors:
3039

Date range:
Fri Dec 01 00:33:23 +0000 2017
Wed Sep 27 09:28:40 +0000 2017


In [3]:
support_authors = (
    df.loc[df["inbound"] == False, "author_id"]
      .value_counts()
)

print("Top 20 support authors:")
print(support_authors.head(20))

Top 20 support authors:
author_id
ChipotleTweets     433
AmazonHelp         410
AppleSupport       318
Uber_Support       189
comcastcares       163
British_Airways    158
VerizonSupport     154
Delta              154
AskPlayStation     153
TMobileHelp        142
SpotifyCares       136
SouthwestAir       127
hulu_support       119
AmericanAir        112
AdobeCare          107
sprintcare          97
Tesco               97
XboxSupport         95
Ask_Spectrum        90
TacoBellTeam        90
Name: count, dtype: int64


In [4]:
brand = "AppleSupport"

brand_df = df[df["author_id"] == brand].copy()

print("Brand:", brand)
print("Rows:", len(brand_df))

print("\nInbound vs outbound:")
print(brand_df["inbound"].value_counts())

print("\nSample messages:")
print(brand_df[["author_id", "inbound", "text"]].head(20).to_string(index=False))

Brand: AppleSupport
Rows: 318

Inbound vs outbound:
inbound
False    318
Name: count, dtype: int64

Sample messages:
   author_id  inbound                                                                                                                                                text
AppleSupport    False                                  @115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.
AppleSupport    False      @115854 Lets take a closer look into this issue. Select the following link to join us in a DM and we'll go from there. https://t.co/GDrqU22YpT
AppleSupport    False                                                                      @115855 Let's go to DM for the next steps. DM us here: https://t.co/GDrqU22YpT
AppleSupport    False                                                                                                @115855 Any steps tried since it started last night?
AppleSupport    False          @1

In [5]:
# Load a larger portion for conversation reconstruction
df_full = pd.read_csv("../data/raw/twcs.csv", nrows=100000)

# Convert date to datetime
df_full["created_at"] = pd.to_datetime(
    df_full["created_at"],
    format="%a %b %d %H:%M:%S %z %Y"
)

# Keep AppleSupport replies
apple_replies = df_full[
    (df_full["author_id"] == "AppleSupport") &
    (df_full["inbound"] == False)
].copy()

print("AppleSupport replies:", len(apple_replies))

AppleSupport replies: 3106


In [6]:
# Find the customer tweet that each AppleSupport reply responds to
customer_ids = apple_replies["in_response_to_tweet_id"].dropna().tolist()

customer_messages = df_full[
    df_full["tweet_id"].isin(customer_ids)
].copy()

print("Customer messages found:", len(customer_messages))

print("\nSample customer messages:")
print(
    customer_messages[
        ["tweet_id", "author_id", "inbound", "text"]
    ].head(20).to_string(index=False)
)

Customer messages found: 3091

Sample customer messages:
 tweet_id author_id  inbound                                                                                                                                                               text
      697    115854     True                                                                                            @AppleSupport The newest update. I️ made sure to download it yesterday.
      698    115854     True                                                                                                                             @AppleSupport  https://t.co/NV0yucs0lB
      702    115855     True                                                                                       @AppleSupport Tried resetting my settings .. restarting my phone .. all that
      704    115855     True                                                                                                   @AppleSupport This is what it looks like https:/

In [7]:
# Create a lookup of tweets by tweet_id
tweet_lookup = df_full.set_index("tweet_id")["text"].to_dict()

# Attach the customer message to each AppleSupport reply
apple_replies["customer_message"] = (
    apple_replies["in_response_to_tweet_id"]
    .map(tweet_lookup)
)

# Keep only replies where we successfully found the customer message
conversation_pairs = apple_replies[
    apple_replies["customer_message"].notna()
][
    ["tweet_id", "customer_message", "text", "created_at"]
].copy()

conversation_pairs.rename(
    columns={
        "tweet_id": "support_tweet_id",
        "text": "support_response"
    },
    inplace=True
)

print("Conversation pairs:", len(conversation_pairs))

print("\nSample conversation pairs:")
print(
    conversation_pairs[
        ["customer_message", "support_response"]
    ].head(10).to_string(index=False)
)

Conversation pairs: 3091

Sample conversation pairs:
                                                                                                                        customer_message                                                                                                                               support_response
                                                                                                  @AppleSupport  https://t.co/NV0yucs0lB                             @115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.
                                                                 @AppleSupport The newest update. I️ made sure to download it yesterday. @115854 Lets take a closer look into this issue. Select the following link to join us in a DM and we'll go from there. https://t.co/GDrqU22YpT
                                                            @AppleSupport Tried resetting my settings .. re

In [8]:
import os

os.makedirs("../data/processed", exist_ok=True)

conversation_pairs.to_csv(
    "../data/processed/apple_support_pairs.csv",
    index=False
)

print("Saved:", len(conversation_pairs), "conversation pairs")
print("File: data/processed/apple_support_pairs.csv")

Saved: 3091 conversation pairs
File: data/processed/apple_support_pairs.csv


In [9]:
print("Unique customer messages:", conversation_pairs["customer_message"].nunique())
print("Unique support responses:", conversation_pairs["support_response"].nunique())

print("\nLongest customer messages:")
print(
    conversation_pairs["customer_message"]
    .str.len()
    .sort_values(ascending=False)
    .head(10)
)

Unique customer messages: 3031
Unique support responses: 3091

Longest customer messages:
97719    312
78524    308
68047    307
88781    306
26744    299
87208    299
73311    299
64970    298
81270    296
94897    294
Name: customer_message, dtype: int64


In [10]:
import re

def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML entities
    text = re.sub(r"&\w+;", " ", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


conversation_pairs["clean_customer_message"] = (
    conversation_pairs["customer_message"]
    .apply(clean_text)
)

print("Sample cleaned messages:")
print(
    conversation_pairs[
        ["customer_message", "clean_customer_message"]
    ].head(20).to_string(index=False)
)

Sample cleaned messages:
                                                                                                                                                  customer_message                                                                                                       clean_customer_message
                                                                                                                            @AppleSupport  https://t.co/NV0yucs0lB                                                                                                                             
                                                                                           @AppleSupport The newest update. I️ made sure to download it yesterday.                                                                    The newest update. I️ made sure to download it yesterday.
                                                                                      @AppleSupport Tried reset

In [11]:
print("Empty messages after cleaning:")

empty_messages = conversation_pairs[
    conversation_pairs["clean_customer_message"].str.len() == 0
]

print(len(empty_messages))

print("\nCustomer message length statistics:")
print(
    conversation_pairs["clean_customer_message"]
    .str.len()
    .describe()
)

Empty messages after cleaning:
22

Customer message length statistics:
count    3091.000000
mean       94.914267
std        59.398360
min         0.000000
25%        51.000000
50%        91.000000
75%       125.000000
max       279.000000
Name: clean_customer_message, dtype: float64


In [12]:
# Remove messages that became empty after cleaning
conversation_pairs = conversation_pairs[
    conversation_pairs["clean_customer_message"].str.len() > 0
].copy()

print("Usable messages:", len(conversation_pairs))

Usable messages: 3069


In [13]:
# Show 100 representative customer messages
sample_messages = (
    conversation_pairs["clean_customer_message"]
    .sample(n=min(100, len(conversation_pairs)), random_state=42)
    .tolist()
)

for i, message in enumerate(sample_messages, 1):
    print(f"{i}. {message}")

1. Sometimes it just slows down and then goes all fast , then some days it just shuts off on me.
2. So annoyed when my Apple Watch tells me to stand so I stand and move around for more than a minute but it still doesn’t want to count. Ahh what more do I need to do
3. hi, since upgrading iOS I can no longer take screenshots. Has this feature been removed?
4. I hate the update to the podcast app on iOS11. The interface is really confusing and unorganized
5. UK
6. I have and my battery life has become even worse
7. I’m going to need you guys to fix my iMessage and FaceTime..it’s not activating...spent an hour on the phone with support
8. When will fix the problem with activation (error 0xE8000013)? Or your phones are now disposable?
9. Yes, I even tried to add another control and its icon appeared on top of the ones that overlap in the bottom left corner.
10. Ayo fix this whole question mark box ting dem shits annoying.
11. my keyboard for my new iPad pro isn’t working. I often have to ta

In [14]:
from collections import Counter

# Display all 3069 usable messages in batches of 100
messages = conversation_pairs["clean_customer_message"].tolist()

for start in range(0, len(messages), 100):
    print(f"\n{'='*80}")
    print(f"MESSAGES {start + 1} - {min(start + 100, len(messages))}")
    print(f"{'='*80}\n")

    for i, message in enumerate(
        messages[start:start + 100],
        start=start + 1
    ):
        print(f"{i}. {message}")


MESSAGES 1 - 100

1. The newest update. I️ made sure to download it yesterday.
2. Tried resetting my settings .. restarting my phone .. all that
3. This is what it looks like
4. I️ have an iPhone 7 Plus and yes I️ do
5. I️ need answers because it’s annoying 🙃
6. Hey and anyone else who upgraded to ios11.1, are y’all having issues with capital “I️” in the Mail app? As it puts in “A”?
7. This is what is happening...
8. Tf is wrong with my keyboard
9. are the call centres closed for the night?
10. hello are all the lines closed for tonight #help
11. I️ upgraded. I️t didn’t work.
12. Hello, internet. Can someone explain why this symbol keeps appearing on my phone and when I️ try to type the letter I️? Also
13. I’ve got a screenshot saying my #iPhoneX is reserved for the 3rd then an email saying it’s the 18th... what happened?
14. Thank you I updated my phone and now it is even slower and barely works. Thank you for ruining my phone.😤
15. I have the iPhone 6s Plus and just did the most rec

In [15]:
sample_300 = (
    conversation_pairs["clean_customer_message"]
    .sample(n=min(300, len(conversation_pairs)), random_state=123)
)

for i, message in enumerate(sample_300, 1):
    print(f"{i}. {message}")

1. hi, since upgrading iOS I can no longer take screenshots. Has this feature been removed?
2. Following this too bc it’s been happening to me!!
3. I-pad air 1
4. yo why is it when I️ type and I️ it makes that damn box? Fix that shit moe
5. Apple really screwed up with this update. The problem when you put I️. My phones constantly freezing and crashing shit sucks.
6. It just shows all message view, not where I can write to that person..
7. needs to figure out what this 👉🏾I️ ..... issue is on my iPhone8
8. MY PHONE IS UNLOCKED SO WHY CAN'T I VIEW MY NOTIFICATIONS #iOS11
9. I can not find an option to get it there. It was just always there when playing.
10. Yayz 11.1 and just 2 English and emoji
11. It’s iOS 11.1.1
12. iPhone 6. Has recently started vibrating randomly.
13. 😒🙄😒🙄 Damn near deleted all of my apps to get an update ... got the update and now all I’m left with are question marks 😐
14. how do I get my keyboard to STOP auto correcting the letter "I️" to a symbol. See attached ph

In [16]:
keywords = {
    "ios_update_issue": [
        "ios", "update", "upgraded", "upgrade", "ios 11"
    ],
    "keyboard_autocorrect_issue": [
        "keyboard", "autocorrect", "auto correct", "typing",
        "type", "letter", "symbol", "question mark box"
    ],
    "device_slow_freezing_crashing": [
        "slow", "slower", "freezing", "freeze", "crash",
        "crashing", "shuts off", "randomly restart"
    ],
    "battery_charging_issue": [
        "battery", "charge", "charging", "percentage"
    ],
    "wifi_mobile_data_issue": [
        "wifi", "wi-fi", "internet", "data", "lte", "connection"
    ],
    "imessage_facetime_issue": [
        "imessage", "facetime"
    ],
    "apple_id_account_issue": [
        "apple id", "account", "recover", "password"
    ],
    "app_or_feature_issue": [
        "app", "icloud", "photos", "podcast", "notes",
        "screenshot", "music"
    ],
    "apple_watch_issue": [
        "watch", "watchos", "apple watch"
    ]
}

for intent, words in keywords.items():
    pattern = "|".join(re.escape(word) for word in words)

    count = conversation_pairs["clean_customer_message"].str.contains(
        pattern,
        case=False,
        na=False
    ).sum()

    print(f"{intent:35} {count}")

ios_update_issue                    926
keyboard_autocorrect_issue          208
device_slow_freezing_crashing       197
battery_charging_issue              267
wifi_mobile_data_issue              127
imessage_facetime_issue             38
apple_id_account_issue              107
app_or_feature_issue                856
apple_watch_issue                   82


In [17]:
# Save the cleaned messages for manual labeling
labeling_data = conversation_pairs[
    ["clean_customer_message", "customer_message", "support_response"]
].copy()

labeling_data.insert(0, "example_id", range(1, len(labeling_data) + 1))

labeling_data.to_csv(
    "../data/processed/apple_support_labeling_data.csv",
    index=False
)

print("Labeling dataset created:", len(labeling_data))

Labeling dataset created: 3069


In [18]:
print(
    labeling_data[
        ["example_id", "clean_customer_message"]
    ].head(20).to_string(index=False)
)

 example_id                                                                                                       clean_customer_message
          1                                                                    The newest update. I️ made sure to download it yesterday.
          2                                                               Tried resetting my settings .. restarting my phone .. all that
          3                                                                                                   This is what it looks like
          4                                                                                       I️ have an iPhone 7 Plus and yes I️ do
          5                                                                                      I️ need answers because it’s annoying 🙃
          6   Hey and anyone else who upgraded to ios11.1, are y’all having issues with capital “I️” in the Mail app? As it puts in “A”?
          7                              

In [19]:
# Create a reproducible 200-example golden evaluation sample

golden_set = labeling_data.sample(
    n=200,
    random_state=42
).copy()

# Add columns that we will manually fill
golden_set["intent"] = ""
golden_set["expected_escalation"] = ""
golden_set["notes"] = ""

golden_set.to_csv(
    "../data/golden/apple_support_golden_set.csv",
    index=False
)

print("Golden set created:", len(golden_set))
print("\nColumns:")
print(golden_set.columns.tolist())

print("\nFirst 10 examples:")
print(
    golden_set[
        ["example_id", "clean_customer_message", "intent",
         "expected_escalation"]
    ].head(10).to_string(index=False)
)


Golden set created: 200

Columns:
['example_id', 'clean_customer_message', 'customer_message', 'support_response', 'intent', 'expected_escalation', 'notes']

First 10 examples:
 example_id                                                                                                                                               clean_customer_message intent expected_escalation
       2652                                                                        Sometimes it just slows down and then goes all fast , then some days it just shuts off on me.                           
       1741 So annoyed when my Apple Watch tells me to stand so I stand and move around for more than a minute but it still doesn’t want to count. Ahh what more do I need to do                           
       2607                                                                             hi, since upgrading iOS I can no longer take screenshots. Has this feature been removed?                           
       

In [2]:
import pandas as pd

In [3]:
golden_set = pd.read_csv(
    "../data/golden/apple_support_golden_set.csv"
)

print("Total examples:", len(golden_set))

print("\nIntent distribution:")
print(golden_set["intent"].value_counts())

print("\nEscalation distribution:")
print(golden_set["expected_escalation"].value_counts())

print("\nMissing labels:")
print(golden_set[["intent", "expected_escalation"]].isnull().sum())

print("\nBlank labels:")
print(
    (golden_set["intent"] == "").sum(),
    "blank intents"
)

print(
    (golden_set["expected_escalation"] == "").sum(),
    "blank escalation labels"
)

Total examples: 200

Intent distribution:
intent
other_or_unclear                 65
app_or_feature_issue             58
battery_charging_issue           21
keyboard_autocorrect_issue       17
device_slow_freezing_crashing    16
ios_update_issue                 11
apple_watch_issue                 4
wifi_mobile_data_issue            4
apple_id_account_issue            3
imessage_facetime_issue           1
Name: count, dtype: int64

Escalation distribution:
expected_escalation
no     115
yes     85
Name: count, dtype: int64

Missing labels:
intent                 0
expected_escalation    0
dtype: int64

Blank labels:
0 blank intents
0 blank escalation labels


In [4]:
import re
import pandas as pd

# Load the full usable AppleSupport dataset we created earlier
labeling_data = pd.read_csv(
    "../data/processed/apple_support_labeling_data.csv"
)

# Load the protected 200-example Golden Set
golden_set = pd.read_csv(
    "../data/golden/apple_support_golden_set.csv"
)

# IDs belonging to the Golden Set
golden_ids = set(golden_set["example_id"])

# Everything except the Golden Set becomes development data
dev_set = labeling_data[
    ~labeling_data["example_id"].isin(golden_ids)
].copy()

print("Total usable messages:", len(labeling_data))
print("Golden Set:", len(golden_set))
print("Development set:", len(dev_set))

Total usable messages: 3069
Golden Set: 200
Development set: 2869


In [5]:
def weak_label(text):
    text = str(text).lower()

    # 1. Keyboard / typing
    if any(word in text for word in [
        "keyboard",
        "autocorrect",
        "auto correct",
        "typing",
        "type ",
        "typed",
        "letter i",
        "question mark",
        "symbol"
    ]):
        return "keyboard_autocorrect_issue"

    # 2. Apple Watch
    if any(word in text for word in [
        "apple watch",
        "watchos",
        "watch os",
        "my watch"
    ]):
        return "apple_watch_issue"

    # 3. iMessage / FaceTime
    if any(word in text for word in [
        "imessage",
        "facetime"
    ]):
        return "imessage_facetime_issue"

    # 4. Apple ID / account
    if any(phrase in text for phrase in [
        "apple id",
        "appleid",
        "password recovery",
        "recover my account",
        "locked out of my account",
        "cannot verify account",
        "can't verify account",
        "account verification"
    ]):
        return "apple_id_account_issue"

    # 5. Battery / charging
    if any(word in text for word in [
        "battery",
        "batteries",
        "charging",
        "charge",
        "battery life",
        "battery drain"
    ]):
        return "battery_charging_issue"

    # 6. Wi-Fi / mobile / Bluetooth connectivity
    if any(word in text for word in [
        "wifi",
        "wi-fi",
        "mobile data",
        "cellular",
        "lte",
        "no service",
        "bluetooth",
        "internet connection",
        "hotspot"
    ]):
        return "wifi_mobile_data_issue"

    # 7. Device performance / freezing / crashing
    if any(word in text for word in [
        "slow",
        "slower",
        "lag",
        "laggy",
        "freeze",
        "freezing",
        "frozen",
        "crash",
        "crashing",
        "shuts off",
        "shut off",
        "restarts",
        "restart itself",
        "reboot"
    ]):
        return "device_slow_freezing_crashing"

    # 8. iOS / software update
    if any(word in text for word in [
        "ios update",
        "ios 11",
        "updated ios",
        "updating ios",
        "update my iphone",
        "software update",
        "software upgrade",
        "upgraded ios",
        "latest ios",
        "new ios"
    ]):
        return "ios_update_issue"

    # 9. Generic Apple app / feature issue
    if any(word in text for word in [
        "app",
        "icloud",
        "itunes",
        "music",
        "podcast",
        "photos",
        "notes",
        "safari",
        "siri",
        "airdrop",
        "notification",
        "home screen",
        "apple pay",
        "face id",
        "touch id",
        "screenshot"
    ]):
        return "app_or_feature_issue"

    # 10. No clear intent
    return "other_or_unclear"


dev_set["intent"] = dev_set["clean_customer_message"].apply(weak_label)

print("\nDevelopment intent distribution:")
print(dev_set["intent"].value_counts())


Development intent distribution:
intent
other_or_unclear                 1293
app_or_feature_issue              557
battery_charging_issue            240
ios_update_issue                  205
device_slow_freezing_crashing     190
keyboard_autocorrect_issue        183
wifi_mobile_data_issue            103
apple_watch_issue                  39
imessage_facetime_issue            34
apple_id_account_issue             25
Name: count, dtype: int64


In [6]:
print("\nSample weakly labelled examples:")

print(
    dev_set[
        ["example_id", "clean_customer_message", "intent"]
    ].sample(30, random_state=42).to_string(index=False)
)


Sample weakly labelled examples:
 example_id                                                                                                                                                                                                                                                   clean_customer_message                        intent
        494                                                                                                                                                                                                   October challenge was fine. November challenge went weird after update              other_or_unclear
       2562                                                                                                                                             hi apple! How am I supposed to enter my password when my number keyboard refuses to work? 😩 happening to me for a while now!    keyboard_autocorrect_issue
        836                                  

In [7]:
dev_set.to_csv(
    "../data/processed/apple_support_dev.csv",
    index=False
)

print("\nSaved development dataset:")
print("../data/processed/apple_support_dev.csv")


Saved development dataset:
../data/processed/apple_support_dev.csv
